In [1]:
!pip install qiskit qiskit-aer qiskit-visualization

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement qiskit-visualization (from versions: none)
ERROR: No matching distribution found for qiskit-visualization


In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer # Corrected import
from qiskit.visualization import plot_histogram
import numpy as np

# Define the number of qubits and the target state
n_qubits = 4
target = '1010'  # Example: search for state |1010>

# Calculate optimal number of iterations
N = 2 ** n_qubits
optimal_iterations = int(np.floor(np.pi / 4 * np.sqrt(N)))

# Initialize quantum circuit
qc = QuantumCircuit(n_qubits, n_qubits)

# Step 1: Initialize qubits in superposition using Hadamard gates
for qubit in range(n_qubits):
    qc.h(qubit)

# Step 2: Define the oracle for the target state
# Oracle flips the phase of the target state (e.g., |1010>)
def create_oracle(qc, n_qubits, target):
    # Apply X gates to flip qubits where target has '0'
    for i, bit in enumerate(target):
        if bit == '0':
            qc.x(i)

    # Multi-controlled Z gate to flip phase of target state
    qc.h(n_qubits - 1)
    qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)  # Multi-controlled X (Toffoli equivalent)
    qc.h(n_qubits - 1)

    # Reverse the X gates
    for i, bit in enumerate(target):
        if bit == '0':
            qc.x(i)

# Step 3: Define the diffusion operator
def create_diffusion(qc, n_qubits):
    # Apply Hadamard gates
    for qubit in range(n_qubits):
        qc.h(qubit)

    # Apply X gates
    for qubit in range(n_qubits):
        qc.x(qubit)

    # Multi-controlled Z gate
    qc.h(n_qubits - 1)
    qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    qc.h(n_qubits - 1)

    # Reverse X gates
    for qubit in range(n_qubits):
        qc.x(qubit)

    # Reverse Hadamard gates
    for qubit in range(n_qubits):
        qc.h(qubit)

# Step 4: Apply Grover's iterations
for _ in range(optimal_iterations):
    create_oracle(qc, n_qubits, target)
    create_diffusion(qc, n_qubits)

# Step 5: Measure all qubits
qc.measure(range(n_qubits), range(n_qubits))

# Step 6: Simulate the circuit
simulator = Aer.get_backend('qasm_simulator')
job = simulator.run(qc, shots=1000) # Corrected to use simulator.run()
result = job.result()
counts = result.get_counts()

# Print and plot results
print("Measurement results:", counts)
plot_histogram(counts)

ModuleNotFoundError: No module named 'qiskit.providers.aer'

In [3]:
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_aer import Aer # Corrected import

# Define the black box function


def oracle(circuit, register, marked_state):
    for i in range(len(marked_state)):
        if marked_state[i] == '1':
            circuit.x(register[i])
    circuit.cz(register[0], register[1])
    for i in range(len(marked_state)):
        if marked_state[i] == '1':
            circuit.x(register[i])

# Define the Grover diffusion operator


def grover_diffusion(circuit, register):
    circuit.h(register)
    circuit.x(register)
    circuit.h(register[1])
    circuit.cx(register[0], register[1])
    circuit.h(register[1])
    circuit.x(register)
    circuit.h(register)

# Define the Grover algorithm


def grover(marked_state):

    # Initialize a quantum register
    # of n qubits
    n = len(marked_state)
    qr = QuantumRegister(n)
    cr = ClassicalRegister(n)
    circuit = QuantumCircuit(qr, cr)

    # Apply the Hadamard gate
    # to each qubit
    circuit.h(qr)

    # Repeat the following procedure
    # O(sqrt(2 ^ n)) times
    num_iterations = int(round((2 ** n) ** 0.5))
    for i in range(num_iterations):
        # Apply the black box function f
        # to the current state to mark
        # the solution
        oracle(circuit, qr, marked_state)

        # Apply the Grover diffusion
        # operator to amplify the amplitude
        # of the marked solution
        grover_diffusion(circuit, qr)

    # Measure the state to obtain
    # a solution x
    circuit.measure(qr, cr)

    # Run the circuit on a simulator
    backend = Aer.get_backend('qasm_simulator')
    job = backend.run(circuit, shots = 1) # Corrected to use backend.run()
    result = job.result()
    counts = result.get_counts()
    x = list(counts.keys())[0]

    return x


# Test the Grover algorithm
marked_state = '101'
result = grover(marked_state)
print(f"The marked state is {result}")

ModuleNotFoundError: No module named 'qiskit.providers.aer'